# 16 · Fresh validation of the solar-context protocol (independently seeded)

**What is validated.** The protocol developed in notebook 14 (`visual_hygrometry.suite.pipeline`), unchanged:
- context candidates L3 / L4 / S / S4, chosen per split on training rows only;
- V clipped to its training range;
- leave-one-season-out over 4 seasons;
- the zero-information test with the **CRT** null generator;
- the season-cluster interval for the design's own G;
- both adequacy auditors, with thresholds frozen in notebook 14 §2.

**Declared before this notebook was first executed.** The worlds, the seeds, the gates and the thresholds below. None of the worlds is a development world, except that the catastrophic season × hour null of notebook 11 is included again on new seeds, as review 5 requires.

| id | gate (100 replicates per world; Monte Carlo SE of a 0.05 rate 0.022) | worlds |
|---|---|---|
| A1 | Type I of the CRT test ≤ 0.085 | every null |
| A2 | power of the CRT test ≥ 0.80 | linear-Gaussian positives with truth ≥ 0.03 |
| A3 | mean estimate ≤ +0.01 in nulls (one-sided: a negative contrast cannot produce a false claim); \|mean estimate − G_design\| ≤ 0.01 in positives | all |
| A4 | the season-cluster 95 % interval covers G_design in ≥ 0.92 of replicates (nominal 0.95) | every world |
| A5 | 90th percentile of \|Δ estimate\| ≤ 0.03 nats after deleting a random 1 % of rows, and ≤ 0.05 after deleting the lowest-D 1 % (full refit) | every world |
| A6 | context-only challenger (τ = 0): false alarms on the chosen context ≤ 0.10; detection with the misspecified L1 ≥ 0.80 | every world / V-N3, V-N4 |
| A7 | repair diagnostic (τ = 0.002 nats per row): false alarms on the chosen context ≤ 0.10; detection with L1 ≥ 0.80 | every world / V-N3, V-N4 |

- G_design is the expected future-season gain of models fitted on three seasons. It is estimated per world by the frozen chronological models on three further seasons, not by E2 (statistical note §2).
- The other null generator, coverage of E2, the week-cluster interval and the chronological season-specific G are reported without gates.
- If A1 fails, no conditional-independence claim may be made with this protocol, and only descriptive predictive results are reported.

In [1]:
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

from functools import partial

import numpy as np
import pandas as pd

from visual_hygrometry import data, suite
from visual_hygrometry.suite import climate, cmi, encodes, gam, regimes, season_x_hour, spec, sun, sun_clock, vegetation

pd.set_option('display.precision', 3)
TAU_AUDIT, TAU_REPAIR = 0.0, 0.002  # frozen in notebook 14 §2
GEN = 'crt'  # frozen in notebook 14 §2

## 1 · Validation worlds

Every world uses a different solar climate from development (heating lag 2.5 h instead of 2, insolation amplitudes 8 K for T and 7 K for D, a stronger dawn minimum).

| world | structure | truth |
|---|---|---|
| V-N1 | climate only | 0 |
| V-N2 | sun-clock image, lagged 1 h, amplitude 1.2 | 0 |
| V-N3 | the notebook 11 catastrophic null: D += 0.8 sin(hour) tanh(season); image encodes it | 0 |
| V-N4 | D += 0.7 cos(hour − 14 h) × (declination − 18°)/4 (diurnal amplitude following day length); image encodes it | 0 |
| V-N5 | D += 0.1 (T − 18) × insolation 1 h earlier; image encodes it | 0 |
| V-N6, V-N7 | moisture residual and image noise persistence 0.90 / 0.97 | 0 |
| V-N8 | independent weather regimes (persistence 0.85 in D, 0.97 in the image) | 0 |
| V-N9 | season-specific senescence and a camera change from season 2 (gain 0.8, offset +0.5) | 0 |
| V-N10 | cloudiness in T, D and image; clear-sky index in C (E2_illum) | 0 |
| V-P1, V-P2 | image sees the moisture residual | 0.03, 0.06 |
| V-P3 | slow shared state | exact |
| V-P4 | shared weather regimes (persistence 0.92) | Monte Carlo |
| V-P5 | V-N10 under E2 (clear-sky index not in C) | exact |

In [2]:
CLIM = partial(climate, t_amp=8.0, t_lag=2.5, d_amp=7.0, d_lag=3.0, dawn=2.0)
day_len = lambda d: 0.7 * np.cos(2 * np.pi * (d.hour - 14) / 24) * (data.solar(d.doy, d.hour, suite.LAT, suite.LON, suite.OFF).solar_decl_deg.to_numpy() - 18) / 4
t_sun = lambda d: 0.1 * (d.temp_cond_c - 18) * sun(d, 1.0)
CLOUD = (-.8, -.9, -.8)
VAL = {'V-N1': dict(spec=spec()), 'V-N2': dict(spec=spec(), modify=sun_clock(1.2, .4, lag=1.0)),
       'V-N3': dict(spec=spec(), modify=encodes(season_x_hour, 1.5, .3), vis=['vis_ctx']),
       'V-N4': dict(spec=spec(), modify=encodes(day_len, 1.5, .3), vis=['vis_ctx']), 'V-N5': dict(spec=spec(), modify=encodes(t_sun, 1.2, .3), vis=['vis_ctx']),
       'V-N6': dict(spec=spec(r_persist=.9, v_persist=.9, r_day=.8, v_day=.8)), 'V-N7': dict(spec=spec(r_persist=.97, v_persist=.97, r_day=.8, v_day=.8)),
       'V-N8': dict(spec=spec(), modify=regimes(.85, .97, mu=-1.5, sd=1.3, nu=.8)),
       'V-N9': dict(spec=spec(), modify=vegetation(amp=(-1.5, -.3), mid=(190, 230), width=10, camera_from=1, gain=.8, offset=.5)),
       'V-N10': dict(spec=spec(cloud=CLOUD, illum=True), illum=True, truth=cmi(spec(cloud=CLOUD, illum=True), ('temp_cond_c', 'clearsky_k'))),
       'V-P1': dict(spec=spec(gamma=gam(.03))), 'V-P2': dict(spec=spec(gamma=gam(.06))), 'V-P3': dict(spec=spec(slow=(1.0, .9))),
       'V-P4': dict(spec=spec(), modify=regimes(.92, .92, mu=-1.5, sd=1.3, nu=.8, shared=True), truth=suite.truth_shared_regime(.5, (0, -1.5), (2, 2.6), (0, .8), seed=1)),
       'V-P5': dict(spec=spec(cloud=CLOUD, illum=True))}
VAL = {k: {'vis': ['vis_moist'], 'climate': CLIM} | W | {'truth': W.get('truth', cmi(W['spec']) if '-P' in k else 0.0)} for k, W in VAL.items()}
print({k: round(W['truth'], 4) for k, W in VAL.items()})
val_raw = suite.run(VAL, 700000, 100)
val = suite.summarise(VAL, val_raw, TAU_AUDIT, TAU_REPAIR)
val

{'V-N1': 0.0, 'V-N2': 0.0, 'V-N3': 0.0, 'V-N4': 0.0, 'V-N5': 0.0, 'V-N6': 0.0, 'V-N7': 0.0, 'V-N8': 0.0, 'V-N9': 0.0, 'V-N10': np.float64(0.0), 'V-P1': np.float64(0.03), 'V-P2': np.float64(0.06), 'V-P3': np.float64(0.0469), 'V-P4': np.float64(0.0081), 'V-P5': np.float64(0.0307)}


,truth,G_design,rej_shift,rej_crt,est,bias,est_minus_G,cover_G,cover_E2,cover_week_G,chrono_G,dest_rand_q90,dest_rare_q90,audit,audit_L1,repair,repair_L1,level,failed
world,,,,,,,,,,,,,,,,,,,
V-N1,0.000e+00,-5.012e-04,0.05,0.02,-3.402e-04,-3.402e-04,1.610e-04,0.89,0.91,0.92,-1.034e-03,1.974e-03,2.245e-03,0.07,0.02,0.00,0.01,2.715,0
V-N2,0.000e+00,-1.116e-03,0.03,0.05,-7.157e-04,-7.157e-04,4.001e-04,0.91,0.99,0.91,-9.434e-04,3.029e-03,2.502e-03,0.09,0.07,0.01,0.00,2.667,0
V-N3,0.000e+00,2.444e-04,0.07,0.08,2.835e-04,2.835e-04,3.915e-05,0.91,0.97,0.92,2.807e-04,1.323e-03,1.679e-03,0.12,0.54,0.00,1.00,2.732,0
V-N4,0.000e+00,1.192e-03,0.04,0.04,6.375e-04,6.375e-04,-5.541e-04,0.78,0.96,0.60,6.120e-04,2.238e-03,2.197e-03,0.19,0.54,0.01,1.00,2.725,0
V-N5,0.000e+00,-2.382e-04,0.06,0.02,-3.195e-04,-3.195e-04,-8.137e-05,1.00,0.98,0.98,-3.322e-04,5.815e-04,3.033e-04,0.15,0.02,0.00,0.00,2.678,0
V-N6,0.000e+00,-2.554e-03,0.15,0.06,-5.131e-03,-5.131e-03,-2.578e-03,0.90,0.92,0.74,-3.368e-03,1.135e-02,1.017e-02,0.08,0.12,0.02,0.01,2.833,0
V-N7,0.000e+00,-1.692e-02,0.15,0.04,-2.414e-02,-2.414e-02,-7.221e-03,0.85,0.96,0.50,-2.105e-02,1.794e-02,1.843e-02,0.09,0.12,0.05,0.02,2.862,0
V-N8,0.000e+00,-5.469e-04,0.08,0.06,-3.740e-04,-3.740e-04,1.729e-04,0.87,0.93,0.88,-4.249e-04,3.222e-03,1.291e-03,0.05,0.01,0.00,0.00,2.792,0
V-N9,0.000e+00,-4.008e-04,0.04,0.03,-6.484e-04,-6.484e-04,-2.476e-04,0.96,0.98,0.95,-6.797e-04,2.636e-03,1.674e-03,0.11,0.00,0.01,0.00,2.735,0


## 2 · Gates

In [3]:
nulls, pos = [k for k in VAL if '-N' in k], [k for k in VAL if '-P' in k]
linpos = [k for k in ['V-P1', 'V-P2', 'V-P3', 'V-P5'] if VAL[k]['truth'] >= .03]
v = lambda w, c: val.loc[w, c]
rej = f'rej_{GEN}'
gate = lambda vals, ok: (np.round(vals, 3), bool(ok))
G = {'A1': gate([v(w, rej) for w in nulls], all(v(w, rej) <= .085 for w in nulls)),
     'A2': gate([v(w, rej) for w in linpos], all(v(w, rej) >= .8 for w in linpos)),
     'A3': gate([v(w, 'est') for w in nulls] + [v(w, 'est_minus_G') for w in pos], all(v(w, 'est') <= .01 for w in nulls) and all(abs(v(w, 'est_minus_G')) <= .01 for w in pos)),
     'A4': gate([v(w, 'cover_G') for w in VAL], all(v(w, 'cover_G') >= .92 for w in VAL)),
     'A5': gate([max(v(w, 'dest_rand_q90'), v(w, 'dest_rare_q90')) for w in VAL], all(v(w, 'dest_rand_q90') <= .03 and v(w, 'dest_rare_q90') <= .05 for w in VAL)),
     'A6': gate([v(w, 'audit') for w in VAL] + [v(w, 'audit_L1') for w in ['V-N3', 'V-N4']], all(v(w, 'audit') <= .1 for w in VAL) and all(v(w, 'audit_L1') >= .8 for w in ['V-N3', 'V-N4'])),
     'A7': gate([v(w, 'repair') for w in VAL] + [v(w, 'repair_L1') for w in ['V-N3', 'V-N4']], all(v(w, 'repair') <= .1 for w in VAL) and all(v(w, 'repair_L1') >= .8 for w in ['V-N3', 'V-N4']))}
pd.DataFrame(G, index=['values', 'pass']).T

,values,pass
A1,"[0.02, 0.05, 0.08, 0.04, 0.02, 0.06, 0.04, 0.0...",True
A2,"[0.99, 1.0, 0.83, 1.0]",True
A3,"[-0.0, -0.001, 0.0, 0.001, -0.0, -0.005, -0.02...",True
A4,"[0.89, 0.91, 0.91, 0.78, 1.0, 0.9, 0.85, 0.87,...",False
A5,"[0.002, 0.003, 0.002, 0.002, 0.001, 0.011, 0.0...",True
A6,"[0.07, 0.09, 0.12, 0.19, 0.15, 0.08, 0.09, 0.0...",False
A7,"[0.0, 0.01, 0.0, 0.01, 0.0, 0.02, 0.05, 0.0, 0...",True


## 3 · Reading

**Outcome: five of seven gates pass; A4 (interval coverage) and A6 (context-only challenger) fail.** No replicate failed (1,500 of 1,500 completed). Both failures were expected from development (notebook 14 §2), and the gates were kept as declared.

**What passes.**
- **A1, the CRT zero-information test.** Type I is 0.02–0.08 in all ten nulls, including:
  - the notebook 11 catastrophic season × hour null (V-N3, 0.08);
  - the day-length-dependent diurnal amplitude (V-N4, 0.04);
  - persistence 0.90 and 0.97 (V-N6 0.06, V-N7 0.04);
  - independent regimes (V-N8 0.06);
  - senescence with a camera change (V-N9 0.03).

  The largest rate, 0.08 in V-N3, is close to the gate. With 100 replicates its one-sided 95 % upper bound is about 0.14, so a modest excess there cannot be ruled out. The ungated shift test fails under persistence (0.15 in V-N6 and V-N7), which confirms the development choice.
- **A2, power.** 0.99 / 1.00 / 0.83 / 1.00 in the linear-Gaussian positives. The CRT is clearly stronger than the shift test in the slow-state world (0.83 against 0.30) and in the shared-regime world (V-P4, truth 0.008, not gated: 0.56 against 0.29).
- **A3, estimates.** Null means lie between −0.024 and +0.001; persistent nulls are negative, as in development. In positives |estimate − G_design| ≤ 0.002. The estimate tracks the design's G, which is 0.003–0.007 below E2 in every positive world.
- **A5, stability.** The 90th percentile change after a 1 % deletion is ≤ 0.018 nats (random and lowest-D).
- **A7, the repair diagnostic (τ = 0.002).** False alarms are ≤ 0.05 in every world, and detection under the misspecified L1 is 1.00 in both V-N3 and V-N4.

**What fails.**
- **A4, the season-cluster interval.** Only 3 of 15 worlds reach 0.92 coverage of G_design (V-N5 1.00, V-N9 0.96, V-N10 0.94).
  - In the other nulls coverage is 0.78–0.91.
  - In positives it is 0.71–0.90; it is lowest with slow or regime information (V-P3 0.76, V-P4 0.71).
  - The bias is small (above), so the intervals are too narrow, not off-centre.
  - The statistical note asks for ≥ 0.93; this notebook's gate was 0.92. The outcome is the same under either threshold.
  - The week-cluster interval is no better (0.50–0.98), and coverage of E2 is lower still in positives (0.61–0.83), because G_design < E2.
  - A t interval over four leave-one-season-out means is therefore not a valid interval for this design. A plausible but untested reason is that any two of the four fold estimates share two of their three training seasons, so they are positively correlated.
- **A6, the context-only challenger.** It fails on both sides:
  - false alarms on the chosen context exceed 0.10 in five worlds (V-N4 0.19, V-N5 0.15, V-N3 0.12, V-N9 0.11, V-P1 0.11);
  - detection under L1 is only 0.54 in V-N3 and V-N4.

  Random Fourier features of the raw context, with fixed length scales, are neither specific nor sensitive enough at this sample size.

**Context repair in a null.** In V-N4, G_design is +0.0012 nats while E2 = 0: the image partly repairs the day-length interaction that the selected context only approximates. This is the mechanism of statistical note §2 (G can exceed E2). It is small here, and the repair diagnostic does not flag it (0.01).

**Consequences, as declared before the run.**
- The protocol does not pass as a whole.
- A1 passed, so the CRT zero-information test is not ruled out for this design. A conditional-independence claim on real data still depends on the CRT sampler describing the real images, which this synthetic suite cannot establish.
- Effect sizes stay descriptive: season-specific G and the point estimate, with no confirmatory interval.
- Adequacy evidence rests on the repair diagnostic. Review 5 allows that only as a secondary diagnostic, and the primary auditor has not been validated.
- Confirmatory Exp 2a stays blocked, and CPER plan step 3 (the imagery download) waits for the review.